# AI Challenge 2026 — Digital Detective

## Решение команды: [ваше название]

**Участники:** [имена]

**Задача:** бинарная сегментация подделок на изображениях.

**Метрика:** AIC Score (гармоническое среднее Dice для positive и (1 − FPR) для negative).

**Подход:**
- Архитектура: UNet + ResNet34 (SMP).
- Лосс: DiceBCELoss (0.4 Dice + 0.6 BCE).
- Аугментации: albumentations.
- Threshold tuning: подбор оптимального порога на валидации.
- Постобработка: `clean_mask()` — удаление мелкого шума.

**Воспроизводимость:**
- Фиксированный `seed=42`.
- Все зависимости в `requirements.txt`.
- Гиперпараметры — в `configs/`.
- Скрипты — в `src/` и `scripts/`.

**Правила конкурса соблюдены:**
- `test.csv` используется **только** для инференса.
- Все внешние данные — из официального датасета.
- Submission: PNG 0/255, размер = оригинал.

In [ ]:
# ============================================================
# НАСТРОЙКА ОКРУЖЕНИЯ
# ============================================================
import json
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from IPython.display import Image, display, Markdown

# Корень проекта
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import (
    read_train_csv, read_test_csv,
    filter_existing_rows, stratified_split,
    TrainDataset, TestDataset,
)
from src.transforms import get_train_transforms, get_val_transforms
from src.model import build_model
from src.losses import DiceBCELoss
from src.metrics import AICMeter, calculate_aic
from src.train import train as train_fn
from src.postprocess import clean_mask
from src.predict import run_predict

# ============================================================
# ВОСПРОИЗВОДИМОСТЬ
# ============================================================
SEED = 42

def set_seed(seed):
    import os
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

# ============================================================
# PATHS & CONFIG
# ============================================================
DATA_DIR = PROJECT_ROOT / 'data'
TRAIN_CSV = DATA_DIR / 'stage1' / 'train.csv'
TEST_CSV = DATA_DIR / 'stage1' / 'test.csv'
LABELS_CSV = DATA_DIR / 'stage1' / 'labels.csv'

CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints'
PRED_DIR = PROJECT_ROOT / 'predictions'
RESULTS_DIR = PROJECT_ROOT / 'results'
SUBMISSION_CSV = PROJECT_ROOT / 'submission.csv'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'plots').mkdir(parents=True, exist_ok=True)

# Гиперпараметры
IMG_SIZE = 384
BATCH_SIZE = 16
NUM_WORKERS = 4
EPOCHS = 25
LR = 1e-4
VAL_RATIO = 0.15
ARCHITECTURE = 'unet'
ENCODER = 'resnet34'
THRESHOLD_START = 0.5
MIN_AREA_RATIO = 0.001

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Project root: {PROJECT_ROOT}')
print(f'Device:       {device}')
if device.type == 'cuda':
    print(f'GPU:          {torch.cuda.get_device_name(0)}')
print(f'Train CSV:    {TRAIN_CSV.exists()}')
print(f'Test CSV:     {TEST_CSV.exists()}')
print(f'Labels CSV:   {LABELS_CSV.exists()}')

## 1. EDA — Анализ данных

Что смотрим:
- Размер датасета (train.csv / test.csv).
- Баланс positive / negative.
- Размеры изображений.
- Примеры подделок с наложенной маской.

In [ ]:
# ============================================================
# ЗАГРУЗКА ДАННЫХ
# ============================================================
rows = read_train_csv(TRAIN_CSV)
print(f'Строк в train.csv: {len(rows)}')

# Фильтруем строки, у которых нет файлов на диске
rows = filter_existing_rows(rows, DATA_DIR, show_progress=True)
print(f'Доступно на диске: {len(rows)}')

# ============================================================
# БАЛАНС POSITIVE / NEGATIVE
# ============================================================
# Источник истины: labels.csv, если он есть. Иначе смотрим содержимое масок.
labels = {}
if LABELS_CSV.exists():
    with open(LABELS_CSV, encoding='utf-8') as f:
        import csv
        for r in csv.DictReader(f):
            labels[r['chng_img_path']] = int(r['is_negative'])
    print(f'Загружено labels.csv: {len(labels)} строк')

if labels:
    n_neg = sum(labels.get(r['chng'], 0) for r in rows)
    n_pos = len(rows) - n_neg
else:
    # Fallback: считаем сами по содержимому масок
    from src.dataset import resolve_path, load_mask_binary
    n_pos, n_neg = 0, 0
    for r in rows[:min(5000, len(rows))]:
        gt = r.get('gt')
        if gt is None:
            n_neg += 1
            continue
        gt_path = resolve_path(gt, DATA_DIR)
        if not gt_path.exists():
            continue
        mask = load_mask_binary(gt_path)
        if mask.sum() == 0:
            n_neg += 1
        else:
            n_pos += 1

print()
print(f'📊 Баланс классов:')
print(f'  Positive: {n_pos}')
print(f'  Negative: {n_neg}')
if n_pos + n_neg > 0:
    print(f'  Доля negative: {100 * n_neg / (n_pos + n_neg):.2f}%')

# Сохраняем для отчёта
eda_summary = {
    'total_csv': len(rows),
    'positive': int(n_pos),
    'negative': int(n_neg),
}
with open(RESULTS_DIR / 'eda_summary.json', 'w', encoding='utf-8') as f:
    json.dump(eda_summary, f, indent=2)

In [ ]:
# ============================================================
# ПРИМЕРЫ ПОДДЕЛОК
# ============================================================
import cv2
from src.dataset import resolve_path, load_rgb, load_mask_binary

# Возьмём 8 random positive
sample = [r for r in rows if r.get('gt') is not None][:8]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, row in enumerate(sample):
    img = load_rgb(resolve_path(row['chng'], DATA_DIR))
    mask = load_mask_binary(resolve_path(row['gt'], DATA_DIR))

    overlay = img.copy()
    if mask.sum() > 0:
        overlay[mask > 0] = [255, 0, 0]
        blended = (0.7 * img + 0.3 * overlay).astype(np.uint8)
    else:
        blended = img

    ax = axes[i // 4, i % 4]
    ax.imshow(blended)
    ax.set_title(f'area={100 * mask.sum() / mask.size:.2f}%', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'plots' / 'samples.png', dpi=100, bbox_inches='tight')
plt.show()

## 2. Архитектура

**UNet + ResNet34** (segmentation_models_pytorch):
- Encoder: ResNet34 с предобученными весами ImageNet.
- Decoder: стандартный UNet-декодер.
- Activation: **None** — модель возвращает сырые логиты (для BCEWithLogitsLoss).
- GFLOPs: ~15.6 (лимит конкурса: ≤ 100).

In [ ]:
# ============================================================
# СБОРКА МОДЕЛИ
# ============================================================
model = build_model(
    architecture=ARCHITECTURE,
    encoder_name=ENCODER,
    encoder_weights='imagenet',
    in_channels=3,
    classes=1,
)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Модель: {ARCHITECTURE} + {ENCODER}')
print(f'Параметров: {n_params:.2f} M')

# FLOPs
try:
    from torch.utils.flop_counter import FlopCounterMode
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    model_cpu = model.cpu().eval()
    try:
        ctx = FlopCounterMode(model_cpu, display=False)
    except TypeError:
        ctx = FlopCounterMode(display=False)
    with ctx as counter, torch.no_grad():
        model_cpu(dummy)
    gflops = counter.get_total_flops() / 1e9
    print(f'GFLOPs @ {IMG_SIZE}x{IMG_SIZE}: {gflops:.2f} (лимит: ≤ 100)')
except Exception as e:
    print(f'FLOPs замер пропущен: {e}')

model = model.to(device)

## 3. Лосс и метрика

### DiceBCELoss
Комбинированный лосс:
- **Dice** — хорошо работает с мелкими объектами, выравнивает вклад масок.
- **BCE** — даёт стабильный градиент.

Веса: `0.4 * Dice + 0.6 * BCE`. BCE чуть больше — в датасете много мелких масок.

### AIC Score
$$\text{AIC} = \frac{2 \cdot \text{Dice}_{pos} \cdot (1 - \text{FPR}_{neg})}{\text{Dice}_{pos} + (1 - \text{FPR}_{neg})}$$

- **Dice_pos** — средний Dice для positive.
- **FPR_neg** — доля negative, где площадь маски ≥ 1% (порог из правил).

In [ ]:
# ============================================================
# ДЕМОНСТРАЦИЯ ЛОССА И МЕТРИКИ
# ============================================================
criterion = DiceBCELoss(weight_dice=0.4, weight_bce=0.6)

# Синтетический пример
targets = torch.zeros(2, 1, 64, 64)
targets[:, :, 10:30, 10:30] = 1

# Идеальные логиты
perfect = (targets * 2 - 1) * 15
# Инвертированные
inverted = -perfect

loss_perfect = criterion(perfect, targets).item()
loss_inverted = criterion(inverted, targets).item()

print(f'Loss (perfect):    {loss_perfect:.6f}')
print(f'Loss (inverted):   {loss_inverted:.6f}')
print()

# Метрика AIC на синтетике
preds_np = torch.sigmoid(perfect).numpy()
targets_np = targets.numpy()
aic = calculate_aic(preds_np, targets_np, pred_threshold=0.5, gt_threshold=0.5)
print(f'AIC (perfect):     {aic:.4f}')

## 4. Обучение

Стратегия:
- Stratified split: 85% train / 15% val.
- Гарантия: в val есть и positive, и negative.
- AMP (mixed precision) — ускорение на GPU.
- ReduceLROnPlateau — снижение LR при стагнации AIC.
- Best model выбирается по максимальному AIC на валидации.

In [ ]:
# ============================================================
# ДАННЫЕ ДЛЯ ОБУЧЕНИЯ
# ============================================================
train_rows, val_rows = stratified_split(
    rows,
    val_ratio=VAL_RATIO,
    seed=SEED,
    labels_path=LABELS_CSV if LABELS_CSV.exists() else None,
)

print(f'Train: {len(train_rows)}')
print(f'Val:   {len(val_rows)}')

# Проверка баланса в val
val_pos = sum(1 for r in val_rows
              if r.get('gt') is not None and labels.get(r['chng'], 0) == 0)
val_neg = len(val_rows) - val_pos
print(f'Val: positive={val_pos}, negative={val_neg}')

train_ds = TrainDataset(train_rows, img_size=IMG_SIZE, train=True, data_dir=DATA_DIR)
val_ds = TrainDataset(val_rows, img_size=IMG_SIZE, train=False, data_dir=DATA_DIR)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

print(f'Батчей train: {len(train_loader)}')
print(f'Батчей val:   {len(val_loader)}')

In [ ]:
# ============================================================
# ОБУЧЕНИЕ
# ============================================================
# Вызываем train из src.train — не дублируем логику
train_fn(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    lr=LR,
    device=device,
    save_dir=str(CHECKPOINT_DIR),
    seed=SEED,
    threshold=THRESHOLD_START,
)

print('✅ Обучение завершено')
print(f'   best.pth: {CHECKPOINT_DIR}/best.pth')

In [ ]:
# ============================================================
# ГРАФИКИ ОБУЧЕНИЯ
# ============================================================
metrics_path = RESULTS_DIR / 'metrics.csv'

if metrics_path.exists():
    df = pd.read_csv(metrics_path)
    print(df.to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss
    axes[0].plot(df['epoch'], df['train_loss'], marker='o', label='Train')
    axes[0].plot(df['epoch'], df['val_loss'], marker='s', label='Val')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss по эпохам')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # AIC
    axes[1].plot(df['epoch'], df['val_aic'], marker='o', color='green')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('AIC Score')
    axes[1].set_title('Val AIC по эпохам')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'plots' / 'training_curves.png',
                dpi=120, bbox_inches='tight')
    plt.show()
else:
    print(f'⚠️ {metrics_path} не найден')

## 5. Threshold Tuning

Подбираем оптимальный порог бинаризации на **валидации**.
Тестовые данные не используются — это соответствует правилам.

Порог влияет на:
- **Dice_pos** — с ростом порога модель консервативнее → Dice падает.
- **FPR_neg** — с ростом порога шума меньше → FPR падает.

Оптимум — точка максимума AIC.

In [ ]:
# ============================================================
# ПРОГОН МОДЕЛИ НА ВАЛИДАЦИИ
# ============================================================
checkpoint = CHECKPOINT_DIR / 'best.pth'
if not checkpoint.exists():
    raise FileNotFoundError(f'Нет {checkpoint}. Сначала обучи модель.')

# Загрузка весов
model_eval = build_model(
    architecture=ARCHITECTURE, encoder_name=ENCODER, encoder_weights=None,
)
ckpt = torch.load(checkpoint, map_location='cpu', weights_only=False)
state = ckpt.get('model_state_dict', ckpt)
model_eval.load_state_dict(state)
model_eval = model_eval.to(device).eval()
print(f'✅ Чекпоинт загружен: {checkpoint}')

# Прогон
all_probs, all_masks = [], []
with torch.inference_mode():
    for batch in val_loader:
        images = batch['image'].to(device)
        masks = batch['mask'].to(device)
        with torch.autocast(
            device_type=device.type, dtype=torch.float16,
            enabled=(device.type == 'cuda'),
        ):
            logits = model_eval(images)
        probs = torch.sigmoid(logits.float()).cpu().numpy()[:, 0]
        all_probs.append(probs)
        all_masks.append(masks.cpu().numpy()[:, 0])

all_probs = np.concatenate(all_probs, axis=0)
all_masks = (np.concatenate(all_masks, axis=0) > 0.5).astype(np.float32)

print(f'Probs shape: {all_probs.shape}')
print(f'Masks shape: {all_masks.shape}')

In [ ]:
# ============================================================
# ПЕРЕБОР ПОРОГОВ
# ============================================================
def compute_aic(probs, masks, threshold):
    """AIC для заданного порога (совпадает с правилами конкурса)."""
    pred_bin = (probs >= threshold).astype(np.float32)

    gt_sums = masks.sum(axis=(1, 2))
    is_pos = gt_sums > 0
    is_neg = ~is_pos

    # Dice for positive
    if is_pos.sum() > 0:
        inter = (pred_bin[is_pos] * masks[is_pos]).sum(axis=(1, 2))
        denom = pred_bin[is_pos].sum(axis=(1, 2)) + masks[is_pos].sum(axis=(1, 2))
        dice = np.where(denom > 0, 2.0 * inter / (denom + 1e-6), 1.0).mean()
    else:
        dice = 0.0

    # 1 - FPR for negative (порог 1% площади)
    if is_neg.sum() > 0:
        h, w = masks.shape[1], masks.shape[2]
        areas = pred_bin[is_neg].sum(axis=(1, 2)) / (h * w)
        fpr = (areas >= 0.01).mean()
        comp2 = 1.0 - fpr
    else:
        comp2 = 1.0

    aic = 2.0 * dice * comp2 / (dice + comp2 + 1e-6) if (dice + comp2) > 0 else 0.0
    return float(aic), float(dice), float(comp2)


thresholds = np.arange(0.30, 0.71, 0.02)
results = []

print(f'{"Threshold":>10} | {"AIC":>8} | {"Dice":>8} | {"1-FPR":>8}')
print('-' * 45)
for th in thresholds:
    aic, dice, comp2 = compute_aic(all_probs, all_masks, th)
    results.append({'threshold': float(th), 'aic': aic, 'dice': dice, '1_minus_fpr': comp2})
    print(f'{th:>10.2f} | {aic:>8.4f} | {dice:>8.4f} | {comp2:>8.4f}')

best = max(results, key=lambda r: r['aic'])
print()
print(f'🏆 Optimal threshold: {best["threshold"]:.2f}')
print(f'   AIC: {best["aic"]:.4f}')

In [ ]:
# ============================================================
# ГРАФИК THRESHOLD VS AIC
# ============================================================
thresholds_list = [r['threshold'] for r in results]
aics = [r['aic'] for r in results]
dices = [r['dice'] for r in results]
comps = [r['1_minus_fpr'] for r in results]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(thresholds_list, aics, marker='o', color='#2c7fb8', linewidth=2)
axes[0].axvline(best['threshold'], color='red', linestyle='--',
                label=f'Optimal: {best["threshold"]:.2f}')
axes[0].set_xlabel('Threshold')
axes[0].set_ylabel('AIC Score')
axes[0].set_title('AIC vs Threshold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(thresholds_list, dices, marker='s', label='Dice (pos)', color='green')
axes[1].plot(thresholds_list, comps, marker='^', label='1-FPR (neg)', color='orange')
axes[1].axvline(best['threshold'], color='red', linestyle='--')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('Component')
axes[1].set_title('Components vs Threshold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'plots' / 'threshold_vs_aic.png',
            dpi=120, bbox_inches='tight')
plt.show()

# Сохраняем оптимальный порог
with open(RESULTS_DIR / 'optimal_threshold.json', 'w', encoding='utf-8') as f:
    json.dump({**best, 'all_results': results}, f, indent=2)

print(f'✅ Сохранено: results/optimal_threshold.json')

## 6. Инференс на тесте

Используем:
- Обученную модель (`best.pth`).
- Оптимальный порог из threshold tuning.
- `clean_mask()` для удаления мелкого шума.
- Ресайз предсказаний к оригинальному размеру (INTER_NEAREST).

**Правила:** `test.csv` используется **только** здесь.

In [ ]:
# ============================================================
# ИНФЕРЕНС
# ============================================================
from types import SimpleNamespace

# Оптимальный порог
with open(RESULTS_DIR / 'optimal_threshold.json', encoding='utf-8') as f:
    opt = json.load(f)
optimal_threshold = opt['optimal_threshold']
print(f'Оптимальный порог: {optimal_threshold:.2f}')

pred_args = SimpleNamespace(
    data_dir=str(DATA_DIR),
    test_csv='stage1/test.csv',
    checkpoint=str(CHECKPOINT_DIR / 'best.pth'),
    architecture=ARCHITECTURE,
    encoder=ENCODER,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    threshold=optimal_threshold,
    min_area_ratio=MIN_AREA_RATIO,
    pred_dir=str(PRED_DIR),
    submission_csv=str(SUBMISSION_CSV),
)

run_predict(pred_args)
print(f'✅ Инференс завершён')
print(f'   submission.csv: {SUBMISSION_CSV}')
print(f'   predictions/:   {PRED_DIR}')

In [ ]:
# ============================================================
# ПРОВЕРКА SUBMISSION
# ============================================================
import csv
from PIL import Image

sub = pd.read_csv(SUBMISSION_CSV)
print(f'Колонки: {sub.columns.tolist()}')
print(f'Строк: {len(sub)}')
print()
print(sub.head(3).to_string())

# Проверки формата
assert set(sub.columns) == {'img_path', 'prediction_path'}, 'Неверные колонки'

n_bad_channel = 0
n_bad_values = 0
n_bad_size = 0

for _, row in sub.head(50).iterrows():
    mask = np.asarray(Image.open(row['prediction_path']))
    if mask.ndim != 2:
        n_bad_channel += 1
    if not set(np.unique(mask)).issubset({0, 255}):
        n_bad_values += 1

for _, row in sub.head(20).iterrows():
    orig_path = DATA_DIR / row['img_path']
    if orig_path.exists():
        orig = Image.open(orig_path)
        mask = Image.open(row['prediction_path'])
        if orig.size != mask.size:
            n_bad_size += 1

print()
print(f'Проверено 50 масок:')
print(f'  Bad channel (не 1-канал): {n_bad_channel}')
print(f'  Bad values (не 0/255):    {n_bad_values}')
print(f'  Bad size (не совпало):    {n_bad_size}')

if n_bad_channel == n_bad_values == n_bad_size == 0:
    print('✅ Submission корректен')

## 7. Финальная сборка

Собираем `submission.zip`:
submission.zip
├── submission.csv
└── predictions/
└── *.png



In [ ]:
# ============================================================
# СБОРКА SUBMISSION.ZIP
# ============================================================
import zipfile

zip_path = PROJECT_ROOT / 'submission.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(SUBMISSION_CSV, arcname='submission.csv')
    for png in PRED_DIR.glob('*.png'):
        z.write(png, arcname=f'predictions/{png.name}')

size_mb = zip_path.stat().st_size / 1024 / 1024
print(f'✅ Собран: {zip_path}')
print(f'   Размер: {size_mb:.2f} МБ')
print(f'   Файлов: {len(list(PRED_DIR.glob("*.png"))) + 1}')

## 📊 Итоги

### Финальные метрики
- **AIC Score на валидации:** см. `results/metrics.csv`.
- **Оптимальный порог:** см. `results/optimal_threshold.json`.
- **Submission:** `submission.zip`.

### Что использовалось
- Архитектура: UNet + ResNet34.
- Лосс: DiceBCELoss (0.4 / 0.6).
- Аугментации: albumentations.
- Postprocessing: `clean_mask(min_area_ratio=0.001)`.

### Воспроизводимость
```bash
pip install -r requirements.txt
python solution.py all